<a href="https://colab.research.google.com/github/joaowinderfeldbussolotto/ml-salary-analysis/blob/feature%2Feda/Desafio_Final_Aprendizado_Maquina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Entendimento do Negócio (CRISP-DM)**
## **Contexto do problema**

O projeto consiste na construção de um pipeline completo de Machine Learning com base em um dataset contendo informações de profissionais no Brasil, incluindo características como idade, experiência, escolaridade, idiomas, região e profissão, com o objetivo de analisar e modelar o comportamento salarial.


---


## **Problema de negócio**

Existe uma necessidade de compreender e prever salários no mercado de trabalho brasileiro, considerando múltiplos fatores individuais e profissionais.

Na prática, isso responde perguntas como:

*   Quais fatores mais influenciam o salário?
*   É possível prever o salário de um profissional?
*   Qual o impacto de educação, experiência e idiomas na renda?
*   Existem desigualdades regionais ou por profissão?


---


## **Objetivo principal**

Desenvolver um modelo de Machine Learning capaz de prever o salário mensal de um profissional com base em suas características.


---


## **Objetivos secundários (valor estratégico)**

Além da predição, o projeto deve gerar insights como:

* Identificar os principais drivers de salário
* Entender o impacto de:
  * Escolaridade
  * Experiência
  * Idiomas
  * Região
  * Profissão
* Detectar padrões e possíveis desigualdades no mercado


---


## **Aplicação no mundo real**

Esse tipo de solução pode ser utilizado por:

* Empresas
  * Definir faixas salariais mais justas
  * Apoiar decisões de contratação
  * Benchmark de mercado
* Profissionais
  * Avaliar se estão sendo bem remunerados
  * Planejar carreira (ex: estudar mais vs trocar de área)
* Consultorias / RH / HRTechs
  * Construção de ferramentas de recomendação salarial
  * Estudos de mercado de trabalho


---

## **Variável alvo (target)**
* Salário (R$) → variável contínua (problema de regressão)


---

## **Variáveis explicativas (features)**
* Idade
* Anos de experiência
* Escolaridade
* Segunda língua
* Terceira língua
* Região
* Profissão


---

## **Tipo de problema de Machine Learning**
**Regressão supervisionada**

Porque:

* Existe variável alvo numérica (salário)
* Queremos prever valores contínuos

---

## **Premissas e riscos do negócio**
* Premissas
  * Os dados representam o mercado de forma razoável
  * As variáveis disponíveis capturam fatores relevantes do salário
* Riscos
  * Dados podem ser **simulados ou não reais** (explicitado no projeto)
  * Possível viés:
    * Regional
    * Profissional
    * Educacional
  * Falta de variáveis importantes:
    * Porte da empresa
    * Setor
    * Localização específica (cidade)

---

## **Critérios de sucesso do projeto**
* **Técnico**
  * Modelo com bom desempenho (ex: baixo RMSE)
  * Comparação entre modelos
Pipeline reprodutível
* **Negócio**
  * Capacidade de explicar:
    * "Por que alguém ganha mais que outro?"
  * Geração de insights claros e acionáveis
* **Produto**
  * Deploy funcional (web app)
  * Interface simples para simulação de salário



---


## **Resumo da Fase**

Este projeto busca transformar dados do mercado de trabalho em inteligência aplicada, permitindo prever salários e entender os principais fatores que influenciam a remuneração, apoiando decisões mais estratégicas tanto para empresas quanto para profissionais.

O problema consiste em prever salários de profissionais no Brasil com base em características como experiência, escolaridade, idiomas e profissão. Além da predição, buscamos entender quais fatores realmente impactam a renda, gerando insights que podem apoiar decisões de carreira e políticas salariais. Trata-se de um problema de regressão supervisionada com aplicação direta em RH, consultorias e planejamento profissional


---



## **Importações e configurações**

In [28]:
import warnings
warnings.filterwarnings("ignore")

import plotly.express as px
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import unicodedata
from IPython.core.interactiveshell import dis
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

LOCAL_PATH = "dataset_salarios_brasil.csv"
REMOTE_PATH = "https://raw.githubusercontent.com/joaowinderfeldbussolotto/ml-salary-analysis/1203f7782b850839356562a40dd8fe6b68405093/dataset_salarios_brasil.csv"

COLUNAS_NUMERICAS = ["Idade", "Anos_Experiencia", "Salario"]
COLUNAS_TEXTUAIS = [
    "Escolaridade",
    "Segunda_Lingua",
    "Terceira_Lingua",
    "Regiao",
    "Profissao"
]

COLUNAS_ESPERADAS = COLUNAS_NUMERICAS + COLUNAS_TEXTUAIS
display(Markdown(f"**Colunas esperadas:** {', '.join(COLUNAS_ESPERADAS)}"))

**Colunas esperadas:** Idade, Anos_Experiencia, Salario, Escolaridade, Segunda_Lingua, Terceira_Lingua, Regiao, Profissao

## **Funções auxiliares**
Remove acentos de uma string.

In [17]:
def remover_acentos(texto):
    if not isinstance(texto, str):
        return texto
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")

## **Carregamento**

Prioriza o arquivo local. Caso não exista ou ocorra erro na leitura,
tenta carregar a partir da URL remota.
Retorna:
- df_: DataFrame carregado
- origem: string indicando a origem do carregamento

In [21]:
# Função de Carregamento
def carregar_dataset(local_path=LOCAL_PATH, remote_path=REMOTE_PATH):
    try:
        df_ = pd.read_csv(local_path)
        origem = f"Arquivo local: {local_path}"
    except Exception as erro_local:
        try:
            df_ = pd.read_csv(remote_path)
            origem = f"URL remota: {remote_path}"
            print(f"Aviso: não foi possível ler o arquivo local. Motivo: {erro_local}")
        except Exception as erro_remoto:
            raise FileNotFoundError(
                "Não foi possível carregar o dataset nem do arquivo local nem da URL remota.\n"
                f"Erro local: {erro_local}\n"
                f"Erro remoto: {erro_remoto}"
            )
    return df_, origem

display(Markdown("**Carregando dataset...**"))
df_raw, origem = carregar_dataset()
display(Markdown(f"**Dataset carregado de {origem}.**"))

**Carregando dataset...**

Aviso: não foi possível ler o arquivo local. Motivo: [Errno 2] No such file or directory: 'dataset_salarios_brasil.csv'


**Dataset carregado de URL remota: https://raw.githubusercontent.com/joaowinderfeldbussolotto/ml-salary-analysis/1203f7782b850839356562a40dd8fe6b68405093/dataset_salarios_brasil.csv.**

## **Padronização**

### **Função de Padronização de Colunas**

Padroniza nomes de colunas para facilitar manipulação no notebook
e no pipeline de Machine Learning.

Regras aplicadas:
- remove espaços nas extremidades;
- remove acentos;
- substitui espaços por underscore;
- remove caracteres especiais;
- remove underscores duplicados;
- remove underscores no início e no fim.

In [32]:
def padronizar_colunas(df_):

    df_ = df_.copy()

    novas_colunas = []
    for col in df_.columns:
        col = str(col).strip()
        col = remover_acentos(col)
        col = col.replace("(R$)", "")
        col = col.replace(" ", "_")
        col = re.sub(r"[^A-Za-z0-9_]", "", col)
        col = re.sub(r"_+", "_", col)
        col = col.strip("_")
        novas_colunas.append(col)

    df_.columns = novas_colunas
    return df_
display(Markdown("**Padronizando colunas...**"))
df_raw = padronizar_colunas(df_raw)
display(Markdown("**Colunas padronizadas.**"))
display(Markdown("**Verificando colunas...**"))
display(Markdown(f"**Colunas esperadas:** {', '.join(COLUNAS_ESPERADAS)}"))
display(Markdown(f"**Colunas encontradas:** {', '.join(df_raw.columns)}"))

**Padronizando colunas...**

**Colunas padronizadas.**

**Verificando colunas...**

**Colunas esperadas:** Idade, Anos_Experiencia, Salario, Escolaridade, Segunda_Lingua, Terceira_Lingua, Regiao, Profissao

**Colunas encontradas:** Idade, Anos_Experiencia, Escolaridade, Segunda_Lingua, Terceira_Lingua, Regiao, Profissao, Salario

### **Função de Padronização de Textos**

Remove espaços extras, transforma valores vazios em NaN e padroniza colunas categóricas.

**Exemplos tratados como ausentes:**

" ", "nan", "None", "null"

In [31]:
def padronizar_texto(df_, colunas_textuais=COLUNAS_TEXTUAIS):

    df_ = df_.copy()

    mapa_ausentes = {
        "": np.nan,
        "nan": np.nan,
        "None": np.nan,
        "none": np.nan,
        "Null": np.nan,
        "null": np.nan,
        "NaN": np.nan
    }

    for col in colunas_textuais:
        if col in df_.columns:
            df_[col] = df_[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
            df_[col] = df_[col].replace(mapa_ausentes)

    return df_

display(Markdown("**Padronizando textos...**"))
df_raw = padronizar_texto(df_raw)
display(Markdown("**Textos padronizados.**"))
display(Markdown("**Verificando textos...**"))
display(Markdown(f"**Colunas esperadas:** {', '.join(COLUNAS_ESPERADAS)}"))
display(Markdown(f"**Colunas encontradas:** {', '.join(df_raw.columns)}"))


**Padronizando textos...**

**Textos padronizados.**

**Verificando textos...**

**Colunas esperadas:** Idade, Anos_Experiencia, Salario, Escolaridade, Segunda_Lingua, Terceira_Lingua, Regiao, Profissao

**Colunas encontradas:** Idade, Anos_Experiencia, Escolaridade, Segunda_Lingua, Terceira_Lingua, Regiao, Profissao, Salario

### **Função de Conversão de Colunas Nunéricas**

Converte colunas numéricas com `errors='coerce'`,
permitindo rastrear valores inválidos posteriormente.

Valores não conversíveis serão transformados em NaN.

In [36]:
def converter_colunas_numericas(df_, colunas_numericas=COLUNAS_NUMERICAS):

    df_ = df_.copy()

    for col in colunas_numericas:
        if col in df_.columns:
            df_[col] = pd.to_numeric(df_[col], errors="coerce")

    return df_

display(Markdown("**Convertendo colunas numéricas...**"))
display(Markdown("**Colunas numéricas convertidas.**"))
display(Markdown("**Verificando colunas numéricas...**"))
display(Markdown(f"**Colunas esperadas:** {', '.join(COLUNAS_ESPERADAS)}"))
display(Markdown(f"**Colunas encontradas:** {', '.join(df_raw.columns)}"))



**Convertendo colunas numéricas...**

**Colunas numéricas convertidas.**

**Verificando colunas numéricas...**

**Colunas esperadas:** Idade, Anos_Experiencia, Salario, Escolaridade, Segunda_Lingua, Terceira_Lingua, Regiao, Profissao

**Colunas encontradas:** Idade, Anos_Experiencia, Escolaridade, Segunda_Lingua, Terceira_Lingua, Regiao, Profissao, Salario

### Função de Validação de Colunas

Verifica se as colunas esperadas estão presentes no dataset.

Retorna um dicionário com:
- colunas faltantes
- colunas extras

In [39]:
def validar_colunas(df_, colunas_esperadas=COLUNAS_ESPERADAS):

    colunas_atuais = set(df_.columns)
    colunas_esperadas = set(colunas_esperadas)

    faltantes = sorted(list(colunas_esperadas - colunas_atuais))
    extras = sorted(list(colunas_atuais - colunas_esperadas))

    return {
        "faltantes": faltantes,
        "extras": extras
    }

display(Markdown("**Colunas Faltantes**"))
display(validar_colunas(df_raw)["faltantes"])
display(Markdown("**Colunas Extras**"))
display(validar_colunas(df_raw)["extras"])

**Colunas Faltantes**

[]

**Colunas Extras**

[]

### **Função de Preparação Inicial**

Aplica as etapas iniciais de preparação sem remover registros:
- padronização de colunas
- padronização de textos
- conversão numérica segura

In [43]:
def preparar_base_inicial(df_):

    df_ = padronizar_colunas(df_)
    df_ = padronizar_texto(df_)
    df_ = converter_colunas_numericas(df_)
    return df_

### **Função de Relatório Inicial de Qualidade**

Gera um relatório inicial de qualidade dos dados após a preparação inicial.

In [49]:
def gerar_relatorio_qualidade(df_raw, df_before, colunas_numericas=COLUNAS_NUMERICAS):

    print("=" * 100)
    print("RELATÓRIO INICIAL DE QUALIDADE DOS DADOS")
    print("=" * 100)

    print("\nDimensão da base:")
    print(f"{df_before.shape[0]:,} linhas x {df_before.shape[1]} colunas")

    print("\nDuplicatas exatas:")
    print(int(df_before.duplicated().sum()))

    print("\nTipos de dados:")
    display(df_before.dtypes.to_frame("tipo"))

    print("\nValores ausentes:")
    ausentes = pd.DataFrame({
        "nulos": df_before.isnull().sum(),
        "%_nulos": (df_before.isnull().mean() * 100).round(2)
    }).sort_values("%_nulos", ascending=False)
    display(ausentes)

    print("\nCardinalidade:")
    cardinalidade = pd.DataFrame({
        "qtd_unicos": df_before.nunique(dropna=False)
    }).sort_values("qtd_unicos", ascending=False)
    display(cardinalidade)

    print("\nValores inválidos convertidos para NaN:")
    relatorio_invalidos = []

    for col in colunas_numericas:
        if col in df_raw.columns and col in df_before.columns:
            serie_original = df_raw[col].astype(str).str.strip()
            serie_convertida = pd.to_numeric(df_raw[col], errors="coerce")

            mascara_invalidos = (
                serie_convertida.isna()
                & serie_original.notna()
                & (serie_original != "")
                & (serie_original.str.lower() != "nan")
            )

            relatorio_invalidos.append({
                "coluna": col,
                "qtd_invalidos": int(mascara_invalidos.sum()),
                "exemplos": df_raw.loc[mascara_invalidos, col].astype(str).unique()[:10].tolist()
            })

    display(pd.DataFrame(relatorio_invalidos))


### **Execução das etapas anteriores**

In [50]:
df_raw, origem_dados = carregar_dataset()

# Base original apenas com nomes de colunas padronizados
df_raw = padronizar_colunas(df_raw)

# Base preparada antes de qualquer remoção de registros
df_before = preparar_base_inicial(df_raw)

# Validação estrutural
validacao = validar_colunas(df_before)

print(f"Origem dos dados: {origem_dados}")
print("\nColunas da base preparada:")
print(df_before.columns.tolist())

print("\nValidação estrutural:")
print(f"Colunas faltantes: {validacao['faltantes']}")
print(f"Colunas extras: {validacao['extras']}")

print("\nPrévia da base preparada:")
display(df_before.head())

# Relatório de qualidade
gerar_relatorio_qualidade(df_raw, df_before)

Aviso: não foi possível ler o arquivo local. Motivo: [Errno 2] No such file or directory: 'dataset_salarios_brasil.csv'
Origem dos dados: URL remota: https://raw.githubusercontent.com/joaowinderfeldbussolotto/ml-salary-analysis/1203f7782b850839356562a40dd8fe6b68405093/dataset_salarios_brasil.csv

Colunas da base preparada:
['Idade', 'Anos_Experiencia', 'Escolaridade', 'Segunda_Lingua', 'Terceira_Lingua', 'Regiao', 'Profissao', 'Salario']

Validação estrutural:
Colunas faltantes: []
Colunas extras: []

Prévia da base preparada:


,Idade,Anos_Experiencia,Escolaridade,Segunda_Lingua,Terceira_Lingua,Regiao,Profissao,Salario
0,56.00,22.00,Superior,Inglês,Italiano,Sudeste,Vendedor,NaN
1,46.00,11.00,Médio,Alemão,Francês,Sudeste,Consultor,NaN
2,32.00,26.00,Fundamental,Inglês,Nenhuma,Sul,Consultor,"6,606.26"
3,60.00,21.00,Pós,Francês,Nenhuma,Nordeste,Policial,"6,673.46"
4,25.00,31.00,Médio,Alemão,Espanhol,Centro-Oeste,Professor,"6,632.40"


RELATÓRIO INICIAL DE QUALIDADE DOS DADOS

Dimensão da base:
15,000 linhas x 8 colunas

Duplicatas exatas:
1

Tipos de dados:


,tipo
Idade,float64
Anos_Experiencia,float64
Escolaridade,object
Segunda_Lingua,object
Terceira_Lingua,object
Regiao,object
Profissao,object
Salario,float64



Valores ausentes:


,nulos,%_nulos
Anos_Experiencia,498,3.32
Salario,497,3.31
Idade,489,3.26
Escolaridade,0,0.00
Segunda_Lingua,0,0.00
Terceira_Lingua,0,0.00
Regiao,0,0.00
Profissao,0,0.00



Cardinalidade:


,qtd_unicos
Salario,14445
Idade,49
Anos_Experiencia,41
Profissao,19
Escolaridade,7
Terceira_Lingua,6
Segunda_Lingua,5
Regiao,5



Valores inválidos convertidos para NaN:


,coluna,qtd_invalidos,exemplos
0,Idade,200,[erro]
1,Anos_Experiencia,200,[xxx]
2,Salario,200,[erro]


## **Diagnóstico de Qualidade dos Dados**

O objetivo não é  "limpar" a base, mas entender a qualidade dela após o carregamento e a padronização.

Essa etapa responde perguntas como:

- Quantas linhas e colunas a base possui?
- Existem duplicatas?
- Quais colunas têm valores ausentes?
- Quais colunas têm muitos valores únicos?
- Houve valores inválidos convertidos para NaN nas colunas numéricas?
- A base já está pronta para a EDA ou ainda exige atenção?


---


O propósito do diagnóstico é gerar uma visão inicial da qualidade da base preparada (df_before), sem ainda remover registros.

Importante porque, no CRISP-DM, antes de explorar padrões e treinar modelos, precisamos saber se os dados são confiáveis, completos e coerentes.



---


Gera um relatório de qualidade dos dados após a etapa de carregamento
e padronização, sem remover registros.

Parâmetros:
- df_raw: base original, apenas com nomes de colunas padronizados
- df_before: base após preparação inicial
- colunas_numericas: lista de colunas numéricas esperadas

Saídas:
- exibe relatórios no notebook
- retorna dicionário com tabelas-resumo para reaproveitamento

In [51]:
def gerar_relatorio_qualidade(df_raw, df_before, colunas_numericas=COLUNAS_NUMERICAS):
    print("=" * 100)
    print("RELATÓRIO DE QUALIDADE DOS DADOS")
    print("=" * 100)

    # Dimensão da base
    print("\nDIMENSÃO DA BASE")
    print("-" * 100)
    print(f"Linhas: {df_before.shape[0]:,}")
    print(f"Colunas: {df_before.shape[1]}")

    # Duplicatas exatas
    print("\nDUPLICATAS EXATAS")
    print("-" * 100)
    qtd_duplicatas = int(df_before.duplicated().sum())
    print(f"Quantidade de duplicatas exatas: {qtd_duplicatas}")

    # Tipos de dados
    print("\nTIPOS DE DADOS")
    print("-" * 100)
    tipos_dados = df_before.dtypes.to_frame("tipo")
    display(tipos_dados)

    # Valores ausentes
    print("\nVALORES AUSENTES")
    print("-" * 100)
    resumo_ausentes = pd.DataFrame({
        "nulos": df_before.isnull().sum(),
        "%_nulos": (df_before.isnull().mean() * 100).round(2)
    }).sort_values("%_nulos", ascending=False)
    display(resumo_ausentes)

    # Cardinalidade
    print("\nCARDINALIDADE DAS COLUNAS")
    print("-" * 100)
    cardinalidade = pd.DataFrame({
        "qtd_unicos": df_before.nunique(dropna=False)
    }).sort_values("qtd_unicos", ascending=False)
    display(cardinalidade)

    # Estatísticas descritivas
    print("\n6. ESTATÍSTICAS DESCRITIVAS — VARIÁVEIS NUMÉRICAS")
    print("-" * 100)
    estatisticas_numericas = df_before[colunas_numericas].describe().T
    display(estatisticas_numericas)

    print("\nESTATÍSTICAS DESCRITIVAS — VARIÁVEIS CATEGÓRICAS")
    print("-" * 100)
    estatisticas_categoricas = df_before.describe(include="object").T
    display(estatisticas_categoricas)

    # Valores inválidos convertidos para NaN
    print("\nVALORES INVÁLIDOS CONVERTIDOS PARA NaN")
    print("-" * 100)

    relatorio_invalidos = []

    for col in colunas_numericas:
        if col in df_raw.columns and col in df_before.columns:
            serie_original = df_raw[col].astype(str).str.strip()
            serie_convertida = pd.to_numeric(df_raw[col], errors="coerce")

            mascara_invalidos = (
                serie_convertida.isna()
                & serie_original.notna()
                & (serie_original != "")
                & (serie_original.str.lower() != "nan")
            )

            qtd_invalidos = int(mascara_invalidos.sum())
            exemplos = df_raw.loc[mascara_invalidos, col].astype(str).unique()[:10].tolist()

            relatorio_invalidos.append({
                "coluna": col,
                "qtd_invalidos": qtd_invalidos,
                "exemplos": exemplos if len(exemplos) > 0 else None
            })

    relatorio_invalidos_df = pd.DataFrame(relatorio_invalidos)
    display(relatorio_invalidos_df)

    # Resumo executivo
    print("\nRESUMO EXECUTIVO DA QUALIDADE")
    print("-" * 100)

    resumo_executivo = pd.DataFrame({
        "indicador": [
            "Quantidade de linhas",
            "Quantidade de colunas",
            "Duplicatas exatas",
            "Total de valores nulos",
            "Colunas com pelo menos 1 nulo",
            "Percentual médio de nulos por coluna"
        ],
        "valor": [
            df_before.shape[0],
            df_before.shape[1],
            qtd_duplicatas,
            int(df_before.isnull().sum().sum()),
            int((df_before.isnull().sum() > 0).sum()),
            round((df_before.isnull().mean() * 100).mean(), 2)
        ]
    })
    display(resumo_executivo)

    # Amostra da base preparada
    print("\nAMOSTRA DA BASE PREPARADA")
    print("-" * 100)
    display(df_before.head(10))

    # Retorno para reaproveitamento
    return {
        "tipos_dados": tipos_dados,
        "resumo_ausentes": resumo_ausentes,
        "cardinalidade": cardinalidade,
        "estatisticas_numericas": estatisticas_numericas,
        "estatisticas_categoricas": estatisticas_categoricas,
        "relatorio_invalidos": relatorio_invalidos_df,
        "resumo_executivo": resumo_executivo
    }

relatorios_qualidade = gerar_relatorio_qualidade(df_raw, df_before)

RELATÓRIO DE QUALIDADE DOS DADOS

DIMENSÃO DA BASE
----------------------------------------------------------------------------------------------------
Linhas: 15,000
Colunas: 8

DUPLICATAS EXATAS
----------------------------------------------------------------------------------------------------
Quantidade de duplicatas exatas: 1

TIPOS DE DADOS
----------------------------------------------------------------------------------------------------


,tipo
Idade,float64
Anos_Experiencia,float64
Escolaridade,object
Segunda_Lingua,object
Terceira_Lingua,object
Regiao,object
Profissao,object
Salario,float64



VALORES AUSENTES
----------------------------------------------------------------------------------------------------


,nulos,%_nulos
Anos_Experiencia,498,3.32
Salario,497,3.31
Idade,489,3.26
Escolaridade,0,0.00
Segunda_Lingua,0,0.00
Terceira_Lingua,0,0.00
Regiao,0,0.00
Profissao,0,0.00



CARDINALIDADE DAS COLUNAS
----------------------------------------------------------------------------------------------------


...

,coluna,qtd_invalidos,exemplos
0,Idade,200,[erro]
1,Anos_Experiencia,200,[xxx]
2,Salario,200,[erro]



RESUMO EXECUTIVO DA QUALIDADE
----------------------------------------------------------------------------------------------------


,indicador,valor
0,Quantidade de linhas,"15,000.00"
1,Quantidade de colunas,8.00
2,Duplicatas exatas,1.00
3,Total de valores nulos,"1,484.00"
4,Colunas com pelo menos 1 nulo,3.00
5,Percentual médio de nulos por coluna,1.24



AMOSTRA DA BASE PREPARADA
----------------------------------------------------------------------------------------------------


,Idade,Anos_Experiencia,Escolaridade,Segunda_Lingua,Terceira_Lingua,Regiao,Profissao,Salario
0,56.00,22.00,Superior,Inglês,Italiano,Sudeste,Vendedor,NaN
1,46.00,11.00,Médio,Alemão,Francês,Sudeste,Consultor,NaN
2,32.00,26.00,Fundamental,Inglês,Nenhuma,Sul,Consultor,"6,606.26"
3,60.00,21.00,Pós,Francês,Nenhuma,Nordeste,Policial,"6,673.46"
4,25.00,31.00,Médio,Alemão,Espanhol,Centro-Oeste,Professor,"6,632.40"
5,38.00,26.00,Médio,Italiano,Nenhuma,Norte,Médico,"12,341.06"
6,56.00,1.00,Médio,Inglês,Inglês,Nordeste,Agrônomo,"3,174.29"
7,36.00,32.00,Médio,Nenhuma,Italiano,Nordeste,Engenheiro,"13,165.88"
8,40.00,25.00,Superior,Inglês,Nenhuma,Centro-Oeste,Motorista,"3,865.02"
9,28.00,27.00,Superior,Inglês,Inglês,Sudeste,Engenheiro,"21,455.03"


## **Regras de Consistência de Negócio**

**Objetivo:**

Garantir que os dados não sejam apenas "tecnicamente válidos", mas também **coerentes com a realidade do mundo real**.

Aqui você responde:

- Esses dados fazem sentido?
- Existe lógica entre idade, experiência e profissão?
- Existem combinações impossíveis ou improváveis?

Aplica regras de consistência de negócio e cria flags de auditoria.

Retorna:
- `df_ `com flags adicionadas
- `resumo_flags`
- lista de colunas de flags

In [55]:
def padronizar_categorias_semanticas(df_):
    """
    Padroniza categorias textuais para evitar problemas de mapeamento
    nas regras de consistência.
    """
    df_ = df_.copy()

    # Padronização de escolaridade
    mapa_escolaridade = {
        "Fundamental": "Ensino Fundamental",
        "Ensino Fundamental": "Ensino Fundamental",

        "Medio": "Ensino Médio",
        "Ensino Medio": "Ensino Médio",
        "Ensino Médio": "Ensino Médio",

        "Tecnico": "Técnico",
        "Técnico": "Técnico",

        "Superior": "Graduação",
        "Graduacao": "Graduação",
        "Graduação": "Graduação",

        "Pos": "Pós-graduação",
        "Pos-graduacao": "Pós-graduação",
        "Pós-graduação": "Pós-graduação",

        "Mestrado": "Mestrado",
        "Doutorado": "Doutorado"
    }

    if "Escolaridade" in df_.columns:
        df_["Escolaridade"] = df_["Escolaridade"].replace(mapa_escolaridade)

    return df_

# Regras de Consistência de Negócio
def aplicar_regras_consistencia(df_):
    """
    Aplica regras de consistência de negócio e cria flags de auditoria.
    """

    df_ = df_.copy()
# Padronização Semântica
df_ = padronizar_categorias_semanticas(df_)
# Parâmetros de Negócio